In [37]:
#importing the libraries with the variables needed

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.python.keras.layers import GlobalMaxPooling2D, Dense

import numpy as np

IMAGE_SIZE = (224, 224) # VGG likes this image size
BATCH_SIZE = 5  # Can be changed to whatever


In [38]:
# Do this if you're storing the data set in drive
# from google.colab import drive
# drive.mount('/content/drive')

In [39]:
# !unzip /content/drive/MyDrive/data.zip

!unzip data.zip -d /content/data

Archive:  data.zip
replace /content/data/glioma/glioma1.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [40]:
import os
import shutil

# Change if your directory is different
# DATA_DIRECTORY = '/content/__MACOSX/data'
DATA_DIRECTORY = '/content/data'
BINARY_DATA_DIRECTORY = '/content/binary_data'

# Change the class folders in data so we only have 2 classes
# Put all the tumors into tumor and no tumor into no tumor

tumor_directory = os.path.join(BINARY_DATA_DIRECTORY, "tumor")
no_tumor_directory = os.path.join(BINARY_DATA_DIRECTORY, "no_tumor")

os.makedirs(tumor_directory, exist_ok=True)
os.makedirs(no_tumor_directory, exist_ok=True)

# put all tumor images into tumor folder
tumor_classes = ["glioma", "meningioma", "pituitary"]
for tumor in tumor_classes:
    tumor_path = os.path.join(DATA_DIRECTORY, tumor)
    for image in os.listdir(tumor_path):
        shutil.copy(os.path.join(tumor_path, image), tumor_directory)

# move all no tumor images into healthy folder
healthy_path = os.path.join(DATA_DIRECTORY, "no_tumor")
for image in os.listdir(healthy_path):
    shutil.copy(os.path.join(healthy_path, image), no_tumor_directory)

print("Data is reorganized into two folders: tumor/ and no_tumor/")

count = 0
for image in os.listdir('/content/binary_data/tumor'):
    count = count + 1
print("tumor image count:", count)

count = 0
for image in os.listdir('/content/binary_data/no_tumor'):
    count = count + 1
print("no tumor image count:", count)

Data is reorganized into two folders: tumor/ and no_tumor/
tumor image count: 8803
no tumor image count: 1757


In [41]:
# Converts the images to ones that VGG accepts, RGB to BGR, etc
# I think this is where we could do the other preprocessing with the images

testing_data = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # required for VGG16 to work
    rotation_range=15,                        # randomly rotate between -15 & +15 deg
    width_shift_range=0.1,                    # shift horizontally up to 10% of its width
    height_shift_range=0.1,                   # shift vertically up to 10% of its height
    zoom_range=0.1,                           # zoom in or out up to 10%
    brightness_range=(0.9, 1.1),              # adjust brightness between 90-110% of original
    horizontal_flip=True                      # flip the image left/right
)

# split data into 80% training & 20% validation
train_generator = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

# load images into training set (80% from original dataset)
train_data = train_generator.flow_from_directory(
    BINARY_DATA_DIRECTORY,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

# load images into validation set (20% from orginal dataset)
val_data = train_generator.flow_from_directory(
    BINARY_DATA_DIRECTORY,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

Found 8449 images belonging to 2 classes.
Found 2111 images belonging to 2 classes.


In [42]:
# automatically computes weights for imbalanced datasets
from sklearn.utils.class_weight import compute_class_weight

# stores the class index of every training image as a list
# e.g. [1, 1, 1, 0, 1, 1, 0, ...]
labels = train_data.classes # 0: no_tumor, 1: tumor

# counts how many images belong to each class, then
# calculates weights so every class contributes equally
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = {
    0: class_weights[0],  # no_tumor
    1: class_weights[1]   # tumor
}

print("Weight of no_tumor class:", class_weights[0])
print("Weight of tumor class   :", class_weights[1])

Weight of no_tumor class: 3.004623044096728
Weight of tumor class   : 0.5998154195655261


In [43]:
# Make new model based on imagenet
base_model = VGG16(weights='imagenet', include_top=False)

# freeze all layers except last 4
# otherwise model training will try to update millions of parameters
# this lets only the last 4 convolutional layers stay trainable
for layer in base_model.layers[:-4]:
    layer.trainable = False

# Flatten, pool, etc
x = base_model.output
x = GlobalMaxPooling2D()(x)
layers = 32 # Can change, probably should add more
x = Dense(layers, activation='relu')(x)
x = Dense(2, activation='sigmoid')(x)
model =

# Add optimizer

# Add loss function , loss function for VGG16 usually


# Add metrics


# Add graphs?

SyntaxError: invalid syntax (ipython-input-2266117916.py, line 16)